# 4.2 RLDA — 회귀 기반 프로파일링과 정보량 추정 · Normal ∥ Masked

## 이 노트북이 답하는 질문

4.0 의 LDA 는 클래스 256개마다 평균을 따로 추정한다. 클래스가 많아지면
(예: 16비트 변수 = 65,536 클래스) 클래스당 학습 레코드가 부족해 학습이 무너진다.

`RLDAClassifier` 는 클래스마다 독립인 평균 대신, **비트의 선형 결합**으로 평균을 모형화한다.

$$ \mu_v \ \approx\ \sum_{b} c_b \cdot \mathrm{bit}_b(v) \ +\ c_0 $$

추정할 계수가 클래스 수가 아니라 **비트 수에 비례**하므로 학습 트레이스가 훨씬 덜 든다.

함께 `RLDAInformationEstimator` 로 **모델이 실제로 얼마나 많은 정보를 담고 있는지**를
비트 단위로 잰다. "공격이 성공했다/실패했다" 보다 정량적인 평가 지표이며,
**대책의 효과를 숫자 하나로 비교**할 수 있다 — 이 노트북에서 Normal 과 Masked 를
비교하는 데 딱 맞는 도구다.

| | |
|---|---|
| 학습 | 타겟별 `/profiling` (동일 예산) |
| 하드웨어 | 불필요. `1.0.SNR` 선행 필요 |

---
## 1. `RLDAClassifier` API

| 단계 | 코드 |
|:----:|------|
| 생성 | `rlda = RLDAClassifier(nb=8, p=4)` |
| 누적 | `rlda.fit_u(traces, x)` — `x` 는 **`(n, nv)` `uint64`** |
| 완료 | `rlda.solve()` |
| 판정 | `rlda.predict_proba(traces, var)` — 변수 번호를 지정한다 |

`nb` 는 변수의 **비트 수**다(클래스 수가 아니다). 바이트 변수면 8.

> **라벨 dtype 이 `uint64` 다.** 다른 API 는 `uint16` 인데 여기만 다르며,
> 학습은 2차원 `(n, nv)`, 뒤에 나올 정보량 추정기는 1차원 `(n,)` 을 받는다.
>
> | API | 라벨 shape | dtype |
> |-----|-----------|-------|
> | `Ttest` | `(n,)` | `uint16` |
> | `SNR`, `Cpa`, `MultiLDA` | `(n, nv)` | `uint16` |
> | `LDAClassifier` | `(n,)` | `uint16` |
> | `RLDAClassifier` | `(n, nv)` | `uint64` |
> | `RLDAInformationEstimator` | `(n,)` | `uint64` |
>
> Trace(트레이스) 배열 `traces`는 어디서나 `(n, ns)` `int16`이다.

> **import 위치 함정:** `RLDAClassifier` 는 `scalib.modeling` 에 있지만
> `RLDAInformationEstimator` 는 **`scalib.metrics`** 에 있다. 모델과 평가 지표가
> 서로 다른 모듈에 있으니 `ImportError` 가 나면 여기를 먼저 본다.

In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.family"] = "NanumGothic"
matplotlib.rcParams["axes.unicode_minus"] = False

from scalib_common import (
    TARGETS, TARGET_IDS, SBOX, AES_BLOCK, sbox_out,
    load_group, load_poi, group_len, dataset_summary, require_target,
)
from scalib.modeling import LDAClassifier, RLDAClassifier
from scalib.metrics import RLDAInformationEstimator   # ← modeling 이 아니다

BYTE = 0
N_ATK = 500
P_DIM = 3

AVAILABLE = {tid: group_len("profiling", tid) for tid in TARGET_IDS
             if require_target(tid)["dataset"].is_file()}
N_PROF = min(AVAILABLE.values())
print("프로파일링 보유:", {TARGETS[t]["short"]: n for t, n in AVAILABLE.items()})
print("→ 양쪽 모두 %d 장으로 맞춘다." % N_PROF)

sets = {}
for tid in AVAILABLE:
    z = load_poi(tid)
    w = z["poi_windows"].astype(int)
    cols = np.unique(w.ravel())
    prof = load_group("profiling", target=tid, n=N_PROF, samples=cols)
    atk = load_group("attack", target=tid, n=N_ATK, samples=cols)
    loc = np.searchsorted(cols, w[BYTE])
    sets[tid] = (prof, atk, loc)
    print("[%s] prof%s  바이트%d 창 %d샘플"
          % (TARGETS[tid]["short"], prof["t"].shape, BYTE, len(loc)))

TRUE_KEY = np.array(sets[list(sets)[0]][1]["attrs"]["fixed_key"], dtype=np.uint8)

프로파일링 보유: {'Normal': 100000, 'Masked': 100000}
→ 양쪽 모두 100000 장으로 맞춘다.


[Normal] prof(100000, 336)  바이트0 창 21샘플


[Masked] prof(100000, 336)  바이트0 창 21샘플


---
## 2. LDA와 나란히 — 학습 트레이스 수를 줄이면

RLDA의 값어치는 **학습 트레이스가 부족할 때** 드러난다. 클래스가 256개이므로 학습 트레이스가 1,000개면
클래스당 레코드가 약 4개이고, 그 정도면 LDA의 클래스 내 산포 행렬이 **특이(singular)**해져
`solve()` 가 실패한다.

RLDA 는 클래스마다 평균을 따로 추정하지 않고 **비트의 선형 결합**으로 모형화하므로
추정할 계수가 훨씬 적고, 같은 조건에서 학습이 성립한다.

아래 표는 그 경계를 직접 보인다. LDA 가 실패하는 지점을 예외로 잡아 표시한다 —
**실패 자체가 결과**이기 때문이다.

In [2]:
GUESS = np.arange(256, dtype=np.uint8)


def key_rank(proba, plaintext_byte, true_k):
    """중간값 확률에서 ``true_k``의 0-based 누적 로그확률 순위를 반환한다.

    ``proba``는 ``(n, 256)``, ``plaintext_byte``는 길이 ``n``이어야 한다. 0 확률은
    로그 발산을 막기 위해 ``1e-300``으로 제한한다. 형상이나 정답 키 인덱스가
    유효하지 않으면 NumPy 예외가 발생하며 입력은 변경하지 않는다.
    """
    lp = np.log(np.maximum(proba, 1e-300))
    v = SBOX[np.bitwise_xor(plaintext_byte[:, None], GUESS[None, :])]
    acc = np.take_along_axis(lp, v, axis=1).sum(axis=0)
    return int(np.where(np.argsort(-acc) == true_k)[0][0])


print("%-8s %8s %16s %16s" % ("타겟", "학습장수", "LDA 순위", "RLDA 순위"))
for tid in sets:
    prof, atk, loc = sets[tid]
    tr_a = np.ascontiguousarray(atk["t"][:, loc])
    tk = int(TRUE_KEY[BYTE])
    for n_p in (500, 1000, 3000, N_PROF):
        n_p = min(n_p, prof["t"].shape[0])
        tr_p = np.ascontiguousarray(prof["t"][:n_p, loc])
        y = sbox_out(prof["p"][:n_p, BYTE], prof["k"][:n_p, BYTE])

        try:
            lda = LDAClassifier(nc=256, p=P_DIM)
            lda.fit_u(tr_p, y.astype(np.uint16))
            lda.solve()
            r_lda = "%d" % key_rank(lda.predict_proba(tr_a), atk["p"][:, BYTE], tk)
        except Exception as e:
            r_lda = "실패(%s)" % type(e).__name__

        try:
            rlda = RLDAClassifier(nb=8, p=P_DIM)
            rlda.fit_u(tr_p, y.astype(np.uint64).reshape(-1, 1))
            rlda.solve()
            r_rlda = "%d" % key_rank(rlda.predict_proba(tr_a, 0), atk["p"][:, BYTE], tk)
        except Exception as e:
            r_rlda = "실패(%s)" % type(e).__name__

        print("%-8s %8d %16s %16s" % (TARGETS[tid]["short"], n_p, r_lda, r_rlda))
    print()
print("LDA 가 예외로 죽는 장수에서도 RLDA 는 답을 낸다 — 그것이 회귀 모형화의 목적이다.")

타겟           학습장수           LDA 순위          RLDA 순위
Normal        500  실패(ScalibError)                0
Normal       1000  실패(ScalibError)                0
Normal       3000                0                0


Normal     100000                0                0

Masked        500  실패(ScalibError)               16
Masked       1000  실패(ScalibError)              111
Masked       3000              169              162


Masked     100000                7               69

LDA 가 예외로 죽는 장수에서도 RLDA 는 답을 낸다 — 그것이 회귀 모형화의 목적이다.


---
## 3. 정보량 추정 — `RLDAInformationEstimator`

"공격이 성공했는가"는 이분법이다. **모델이 Trace 한 장에서 몇 비트를 건지는가**를 알면
구현끼리, 대책 적용 전후를 정량적으로 비교할 수 있다.

절차는 두 단계다.

1. `rlda.get_clustered_model(var, t)` — 계산을 감당할 수 있게 클래스를 묶은 간이 모델
2. `RLDAInformationEstimator(model, max_popped_classes)` → `fit_u` → `get_information()`

`get_information()` 은 **(하한, 상한)** 튜플을 준다. 최대 8비트(바이트 변수)이며,
클수록 그 모델이 Trace에서 뽑아내는 정보가 많다는 뜻이다.

> **음수가 나올 수 있다.** 이 값은 순수한 상호정보량이 아니라 **모델이 맞다고 가정했을 때**
> 얻는 정보(perceived information)다. 모델이 실제 누설과 전혀 맞지 않으면 음수가 된다.
> 해석은 단순하다 — **0 이하이면 얻은 정보가 없다.**

**이 지표가 이 노트북의 결론이다.** Normal 과 Masked 를 "복구했다/못했다" 가 아니라
비트 수로 비교할 수 있다.

> **`get_deviation()` 은 호출하지 않는다.** scalib 0.6.4 에서 내부적으로 예외가 아닌 값을
> `raise` 해 `TypeError: exceptions must derive from BaseException` 가 난다(업스트림 버그).

In [3]:
T_CLUSTER = 1.0
MAX_POPPED = 512

info = {}
for tid in sets:
    prof, atk, loc = sets[tid]
    tr_p = np.ascontiguousarray(prof["t"][:, loc])
    tr_a = np.ascontiguousarray(atk["t"][:, loc])
    y_p = sbox_out(prof["p"][:, BYTE], prof["k"][:, BYTE]).astype(np.uint64)
    y_a = sbox_out(atk["p"][:, BYTE], TRUE_KEY[BYTE]).astype(np.uint64)

    rlda = RLDAClassifier(nb=8, p=P_DIM)
    rlda.fit_u(tr_p, y_p.reshape(-1, 1))
    rlda.solve()
    try:
        model = rlda.get_clustered_model(0, T_CLUSTER)
        est = RLDAInformationEstimator(model, MAX_POPPED)
        est.fit_u(tr_a, y_a)                      # 라벨은 1차원 uint64
        lo, hi = est.get_information()
        info[tid] = (float(lo), float(hi))
        print("[%-8s] 정보량 %.4f ~ %.4f 비트  (변수 8비트 중)"
              % (TARGETS[tid]["short"], lo, hi))
    except Exception as e:
        print("[%-8s] 정보량 추정 실패: %s: %s"
              % (TARGETS[tid]["short"], type(e).__name__, e))

print()
if len(info) == 2:
    n_lo, n_hi = info["tiny-AES-c"]
    m_lo, m_hi = info["masked-aes-c"]
    print("[비교] 파형 한 장이 SBox 출력 바이트에 대해 주는 정보")
    print("  Normal %.4f ~ %.4f 비트" % (n_lo, n_hi))
    print("  Masked %.4f ~ %.4f 비트" % (m_lo, m_hi))
    print()
    print("  0 비트 = 파형을 봐도 그 값에 대해 아무것도 모른다는 뜻이다.")
    print("  이 숫자 하나로 '대책이 얼마나 효과가 있었는가' 를 말할 수 있다 —")
    print("  '복구 성공/실패' 보다 훨씬 정량적이다.")
    if m_lo < 0:
        print()
        print("  Masked 값이 **음수**로 나왔다. 상호정보량은 음수가 될 수 없으므로")
        print("  이것은 '정보가 마이너스' 라는 뜻이 아니라, 이 추정량이 **모델 기반**")
        print("  이라는 데서 온다. 추정량은 실제로 '모델이 맞다고 가정했을 때 얻는")
        print("  정보'(perceived information)이고, 모델이 데이터와 전혀 안 맞으면")
        print("  음수가 나올 수 있다. 읽는 법은 간단하다 — **0 이하 = 얻은 정보 없음**.")
        print("  Masked 에서 RLDA 모델이 실제 누설을 전혀 설명하지 못한다는 뜻이다.")

[Normal  ] 정보량 1.3064 ~ 1.3064 비트  (변수 8비트 중)
[Masked  ] 정보량 0.0005 ~ 0.0005 비트  (변수 8비트 중)

[비교] 파형 한 장이 SBox 출력 바이트에 대해 주는 정보
  Normal 1.3064 ~ 1.3064 비트
  Masked 0.0005 ~ 0.0005 비트

  0 비트 = 파형을 봐도 그 값에 대해 아무것도 모른다는 뜻이다.
  이 숫자 하나로 '대책이 얼마나 효과가 있었는가' 를 말할 수 있다 —
  '복구 성공/실패' 보다 훨씬 정량적이다.


### 클러스터 임계 `t` 의 영향

`t` 가 작으면 클래스를 더 잘게 유지해 정확하지만 느리고, 크면 많이 묶어 빠르지만 거칠다.
값을 바꿔 가며 추정이 안정적인지 확인한다.

In [4]:
print("%-8s %8s %10s %10s" % ("타겟", "t", "하한", "상한"))
for tid in sets:
    prof, atk, loc = sets[tid]
    tr_p = np.ascontiguousarray(prof["t"][:, loc])
    tr_a = np.ascontiguousarray(atk["t"][:, loc])
    y_p = sbox_out(prof["p"][:, BYTE], prof["k"][:, BYTE]).astype(np.uint64)
    y_a = sbox_out(atk["p"][:, BYTE], TRUE_KEY[BYTE]).astype(np.uint64)

    rlda = RLDAClassifier(nb=8, p=P_DIM)
    rlda.fit_u(tr_p, y_p.reshape(-1, 1))
    rlda.solve()
    for t in (0.5, 1.0, 2.0):
        try:
            est = RLDAInformationEstimator(rlda.get_clustered_model(0, t), MAX_POPPED)
            est.fit_u(tr_a, y_a)
            lo, hi = est.get_information()
            print("%-8s %8.1f %10.4f %10.4f" % (TARGETS[tid]["short"], t, lo, hi))
        except Exception as e:
            print("%-8s %8.1f   실패: %s" % (TARGETS[tid]["short"], t, type(e).__name__))
    print()

타겟              t         하한         상한


Normal        0.5     1.3064     1.3064
Normal        1.0     1.3064     1.3064
Normal        2.0     1.3064     1.3064



Masked        0.5     0.0005     0.0005
Masked        1.0     0.0005     0.0005
Masked        2.0     0.0005     0.0005



---
## 4. 요약

| 항목 | 내용 |
|------|------|
| 목적 | 클래스가 많을 때도 견디는 **회귀 기반** 프로파일링, 그리고 **정보량** 정량화 |
| 모델 | `RLDAClassifier(nb, p)` → `fit_u(traces, x)` → `solve()` → `predict_proba(traces, var)` |
| `x` | 학습은 `(n, nv)` **`uint64`** |
| 정보량 | `get_clustered_model(var, t)` → `RLDAInformationEstimator(...)` → `get_information()` → (하한, 상한) 비트 |
| import | `RLDAClassifier` 는 `scalib.modeling`, `RLDAInformationEstimator` 는 **`scalib.metrics`** |
| 주의 | `RLDAInformationEstimator.fit_u` 의 라벨은 **1차원** `uint64`. `get_deviation()` 은 0.6.4 에서 버그 |

### 이 Dataset(데이터셋)에서 관측한 것

- 학습 트레이스 수가 적을 때 LDA는 예외로 종료되고 RLDA는 결과를 낸다.
- 정보량으로 보면 Normal 과 Masked 의 차이가 비트 단위로 드러난다.
  마스킹의 효과를 "성공/실패" 가 아니라 **연속적인 값**으로 말할 수 있다.

### 실패 시 점검

1. `TypeError: argument 'label' ... cannot be converted` → 라벨 shape/dtype 을 위 표대로 맞춘다.
2. `ImportError` → `RLDAInformationEstimator` 는 `scalib.metrics` 에 있다.
3. 정보량이 0 근처 → POI나 학습 트레이스 수를 점검한다. **대책 때문일 수도 있다.**

다음: `5.0.SASCA.ipynb` — 여러 중간값의 정보를 그래프로 합친다.